Evaluation modes: 

1. Interact with system -> collect logs -> {$Q_j$} 
2. Generate data (from answer generate questions, e.g. 5 questions per answer): 

    ($Q_i$, $A_i$) - Real Data (FAQ)
    $A_i$ -> generate -> $Q_{i1}$, ..., $Q_{i5}$

    then for each of the generated questions we ask the RAG system those questions, which will generate five answers. We will compare answers with the original, correct one. 
    Also, does the answer belong to the retrieved documents in database by the RAG system? 

In [3]:
from src.ingest import load_faq_data
documents = load_faq_data()  

In [4]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [5]:
documents_llm = []
for doc in documents: 
    if doc["course"] == "llm-zoomcamp": 
        documents_llm.append(doc) 

len(documents_llm)

85

In [6]:
documents = documents_llm 

In [7]:
doc = documents[0] 
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [8]:
from pydantic import BaseModel 

class Questions(BaseModel): 
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI 

load_dotenv()
openai_client = OpenAI()

In [11]:
import json 
user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions}, 
    {"role": "user", "content": user_prompt}
]

OpenaAI client `parse` function helps specifying how to enforce a specific structure (in this case, the Questions format) to the output. 

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini", 
    input=messages, 
    text_format=Questions, 
)

In [15]:
response.output_parsed.questions

['I just found this course a bit late — can I still sign up and follow along?',
 'Is it too late to join the course if I missed the start?',
 'Can I still take part in the course even though it already started?',
 'If I join now, am I still eligible for a certificate or is it too late for that?',
 'What do I need to do to get a certificate if I’m joining after the course has already begun?']

In [17]:
from src.evaluation_utils import llm_structured

In [18]:
result, usage = llm_structured(
    openai_client, 
    data_gen_instructions, 
    user_prompt, 
    Questions
)

print(result.questions)

['I just found this course — can I still sign up and join in, or is it too late?', 'If I join the course late, do I still get a certificate, or is that only for people who started on time?', 'What do I need to do to be eligible for the course certificate if I’m joining after it already started?', 'Is there a deadline for project submission if I want the certificate, or can I send it anytime?', 'I missed the start of the course — can I still participate, and what’s the latest I can submit my project for a certificate?']


In [19]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=131, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=338)

In [20]:
from src.evaluation_utils import calc_price

In [21]:
calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.0005895000000000001,
 'total_cost': 0.0007447500000000001}

In [22]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — can I still sign up and join in, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course late, do I still get a certificate, or is that only for people who started on time?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the course certificate if I’m joining after it already started?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for project submission if I want the certificate, or can I send it anytime?',
  'document': '74eb249bbf'},
 {'question': 'I missed the start of the course — can I still participate, and what’s the latest I can submit my project for a certificate?',
  'document': '74eb249bbf'}]

In [23]:
import pandas as pd 
pd.DataFrame(records)

,question,document
0,I just found this course — can I still sign up...,74eb249bbf
1,"If I join the course late, do I still get a ce...",74eb249bbf
2,What do I need to do to be eligible for the co...,74eb249bbf
3,Is there a deadline for project submission if ...,74eb249bbf
4,I missed the start of the course — can I still...,74eb249bbf


Let's do this for all documents

In [24]:
from src.evaluation_utils import llm_structured_retry 

The following function processes one document (creates prompts and then generates answers + collects usage), as done so far but in a function: 

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

This will iterate over the first 5 documents: 

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

It works, but it processes one document at a time (sequentially), which can be very slow. 

Alternative: process documents *in parallel*. You need to be careful with rate limits. 

In [26]:
from concurrent.futures import ThreadPoolExecutor
from src.evaluation_utils import map_progress

In [27]:
with ThreadPoolExecutor(max_workers=6) as pool: 
    results = map_progress(pool, documents, generate_ground_truth, )

  0%|          | 0/85 [00:00<?, ?it/s]

In [28]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

425

In [29]:
results[1]

([{'question': 'I signed up for the LLM Zoomcamp, but I never got a confirmation email. Do I still need to wait for one, or can I just start the course?',
   'document': '977bf7786c'},
  {'question': 'Is there any actual accepted list for the LLM Zoomcamp, or is registration only for measuring interest before it starts?',
   'document': '977bf7786c'},
  {'question': 'Do I have to register first before I can watch the lessons and submit homework, or can I begin right away while the homework form is open?',
   'document': '977bf7786c'},
  {'question': 'If I already registered for the LLM Zoomcamp, does that change anything for access, or is it just for the organizers to see how many people are interested?',
   'document': '977bf7786c'},
  {'question': 'Should I expect a confirmation message after signing up for the LLM Zoomcamp, or is registering not required to participate?',
   'document': '977bf7786c'}],
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_t

Calculate total cost: 

In [31]:
from src.evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.06337950000000002

Equivalent function: 

In [33]:
from src.evaluation_utils import calc_total_price
calc_total_price(usages)

0.06337950000000002

In [34]:
import pandas as pd
df_ground_truth = pd.DataFrame(ground_truth)

In [35]:
df_ground_truth

,question,document
0,I just found this course — is it still possibl...,74eb249bbf
1,"Can latecomers still enroll, or is it too late...",74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,What do I need to do in order to be eligible f...,74eb249bbf
4,"If I missed the beginning, can I still complet...",74eb249bbf
...,...,...
420,Why does pip keep giving me requests 2.28 when...,4b30b918bc
421,How can I install requests straight from GitHu...,4b30b918bc
422,I’m getting a 401 Client Error while setting t...,4b30b918bc
423,What’s the exact pip command to pull the fixed...,4b30b918bc


In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

To load the prepared file directly from the course repo: 

In [41]:
# !PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main
# !wget -O data/ground_truth-new.csv ${PREFIX}/04-evaluation/data/ground_truth-new.csv

In [ ]:
# !PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main
# !wget -O data/ground-truth-data.csv ${PREFIX}/04-evaluation/data/ground-truth-data.csv

/04-evaluation/data/ground-truth-data.csv: Scheme missing.
